## ExPECA Testbed HI Setup

Start with reading README to understand the purpose of this notebook and the setup options.

---

### Overview

**The notebook will create a setup with the following components**

- Edge Device: Start a containerized application that runs the edge device in the ExPECA testbed.
  - Option 1 = Raspberry Pi (arm64)
  - Option 2 = Normal testbed node (amd64)
- Edge Server: Start a containerized application that runs the edge server in the ExPECA testbed.

**Steps to follow:**

1. Setup authentication for the ExPECA testbed.
2. Install required packages and libraries
3. Setup Edge Server
  - Reserve testbed equiment
  - Load and start container
4. Setup Edge Device
  - Select wanted option
  - Reserve testbed equiment
  - Load and start container


### 1. Setup Authentication for the ExPECA Testbed

---

**Steps:**

1. Login to Chameleon and download openrc.sh file from [here](https://testbed.expeca.proj.kth.se/project/api_access/openrc/).
2. Upload it here next to this notebook and continue.
3. Edit path to match your "xxx-project-openrc.sh" file if needed.
4. Enter your ExPECA password when asked.

In [1]:
import os, re
from getpass import getpass

# Replace with your OpenStack credentials file path
with open('iustin-project-openrc.sh', 'r') as f:
    script_content = f.read()
    pattern = r'export\s+(\w+)\s*=\s*("[^"]+"|[^"\n]+)'
    matches = re.findall(pattern, script_content)

    for name, value in matches:
        os.environ[name] = value.strip('"')

password = getpass('enter your expeca password:')
os.environ['OS_PASSWORD'] = password


### 2. Install Required Packages and Libraries

---

There might be some warnings, ignore them and test if everything still works.

In [2]:
# Install notebook dependencies from the project terminal before this cell:
#   scripts/setup_expeca_notebook_env.sh
# Then restart this notebook kernel.

# Import required libraries
import json, time
try:
    from loguru import logger
    import chi.network, chi.container, chi.network
    from chi.expeca import reserve, list_reservations, unreserve_byid, get_container_status, wait_until_container_removed, show_reservation_byname, restart_sdr, make_sdr_ni, make_sdr_mango, sdr_tools, get_available_publicips, get_segment_ids, get_radio_interfaces, get_worker_interfaces
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Missing ExPECA notebook dependencies. Run "
        "scripts/setup_expeca_notebook_env.sh from the project terminal, "
        "then restart this notebook kernel."
    ) from exc


# Set these to the images you built and pushed for ExPECA.
# CPU baseline: EDGE_SERVER_IMAGE_TAG="cpu-amd64-001", EDGE_SERVER_DEVICE="cpu"
# GPU server:   EDGE_SERVER_IMAGE_TAG="gpu-amd64-001", EDGE_SERVER_DEVICE="cuda"
EXPECA_IMAGE_NAMESPACE = "justin157"
EDGE_SERVER_IMAGE_TAG = "cpu-amd64-001"
EDGE_DEVICE_AMD64_IMAGE_TAG = "cpu-amd64-001"
EDGE_DEVICE_ARM64_IMAGE_TAG = "cpu-arm64-001"
EDGE_SERVER_DEVICE = "cpu"
EDGE_DEVICE_DEVICE = "cpu"
EDGE_SERVER_IMAGE = f"{EXPECA_IMAGE_NAMESPACE}/hi-framework-edge-server:{EDGE_SERVER_IMAGE_TAG}"
EDGE_DEVICE_AMD64_IMAGE = f"{EXPECA_IMAGE_NAMESPACE}/hi-framework-edge-device:{EDGE_DEVICE_AMD64_IMAGE_TAG}"
EDGE_DEVICE_ARM64_IMAGE = f"{EXPECA_IMAGE_NAMESPACE}/hi-framework-edge-device:{EDGE_DEVICE_ARM64_IMAGE_TAG}"

def containers_named(container_name):
    """Return all ExPECA containers matching this name."""
    return [container for container in chi.container.list_containers() if container.name == container_name]


def print_container_summary(container):
    print(
        "name=", container.name,
        "uuid=", container.uuid,
        "status=", container.status,
        "reason=", getattr(container, "status_reason", None),
        "image=", getattr(container, "image", None),
        "addresses=", getattr(container, "addresses", None),
    )


def print_named_containers(container_name):
    matches = containers_named(container_name)
    if not matches:
        print(f"No containers named {container_name}.")
        return matches

    print(f"Containers named {container_name}:")
    for container in matches:
        print_container_summary(container)
    return matches


def wait_for_containers_removed(container_uuids, poll_interval=5, max_wait_seconds=180):
    deadline = time.time() + max_wait_seconds
    remaining = set(container_uuids)
    while remaining:
        existing = {
            container.uuid
            for container in chi.container.list_containers()
            if container.status not in {"Deleted", "Error"}
        }
        remaining = remaining & existing
        if not remaining:
            print("Requested containers are removed.")
            return
        if time.time() >= deadline:
            raise TimeoutError(f"Timed out waiting for removal: {sorted(remaining)}")
        print("Waiting for removal:", sorted(remaining))
        time.sleep(poll_interval)


def destroy_named_containers(container_name, statuses=None, wait=True):
    """Destroy containers by name. Optionally restrict to statuses such as {'Error', 'Creating'}."""
    matches = containers_named(container_name)
    if statuses is not None:
        matches = [container for container in matches if container.status in statuses]

    if not matches:
        print(f"No matching containers to destroy for {container_name}.")
        return []

    destroyed = []
    for container in matches:
        print("Destroying", container.name, container.uuid, container.status)
        chi.container.destroy_container(container.uuid)
        destroyed.append(container.uuid)

    if wait:
        wait_for_containers_removed(destroyed)
    return destroyed


def inspect_named_container(container_name, include_logs=True):
    matches = print_named_containers(container_name)
    if len(matches) != 1:
        if len(matches) > 1:
            print("Multiple matches exist. Clean up stale duplicates before using name-based cells.")
        return matches

    container = matches[0]
    if include_logs:
        print("logs:")
        try:
            print(chi.container.get_logs(container.uuid))
        except Exception as exc:
            print("Could not fetch logs yet:", repr(exc))
    return matches


def single_container_ref(container_name):
    matches = containers_named(container_name)
    if len(matches) == 1:
        return matches[0].uuid
    if len(matches) > 1:
        print_named_containers(container_name)
        raise RuntimeError(
            f"Multiple containers named {container_name}. "
            "Destroy stale duplicates by UUID/name before creating or polling."
        )
    raise RuntimeError(f"No container named {container_name} was found.")


def create_container_with_progress(container_name, create_kwargs, poll_interval=15, max_wait_seconds=1200):
    """Create an ExPECA container and print status until it is Running or Error."""
    existing = [
        container
        for container in containers_named(container_name)
        if container.status not in {"Deleted", "Error"}
    ]
    if existing:
        print_named_containers(container_name)
        raise RuntimeError(
            f"Container name {container_name} already exists. "
            "If it is already Running, skip this creation cell and continue. "
            "If you want a fresh container, run the cleanup cell and destroy it first."
        )

    container_ref = None
    try:
        container = chi.container.create_container(
            name=container_name,
            start_timeout=60,
            **create_kwargs,
        )
        container_ref = container.uuid
    except TimeoutError:
        logger.warning(f"{container_name} is still creating after 60s; polling status now.")
        container_ref = single_container_ref(container_name)
    except Exception as exc:
        matches = containers_named(container_name)
        if not matches:
            raise
        logger.warning(f"create_container returned {type(exc).__name__}: {exc}. Polling existing container.")
        if len(matches) > 1:
            print_named_containers(container_name)
            raise RuntimeError(f"Multiple containers named {container_name}; clean up duplicates before continuing.") from exc
        container_ref = matches[0].uuid

    deadline = time.time() + max_wait_seconds
    while True:
        container = chi.container.get_container(container_ref)
        reason = getattr(container, "status_reason", None)
        addresses = getattr(container, "addresses", None)
        print(time.strftime("%H:%M:%S"), "status:", container.status, "reason:", reason, "addresses:", addresses)

        if container.status == "Running":
            logger.success(f"{container_name} is running.")
            return container
        if container.status in {"Error", "Deleted"}:
            raise RuntimeError(f"{container_name} failed with status {container.status}: {reason}")
        if time.time() >= deadline:
            raise TimeoutError(f"Timed out waiting for {container_name}; last status was {container.status}")

        time.sleep(poll_interval)


### Cleanup stale containers

If a previous run was interrupted, ExPECA may keep `hi-edge-server` or `hi-edge-device` containers around. Run the inspection cell below before creating containers again. Destroy only stale containers you no longer need.

In [3]:
# Inspect existing containers before creating containers again.
print_named_containers("hi-edge-server")
print_named_containers("hi-edge-device")

# Common recovery after an interrupted run: remove stale non-running duplicates.
# Run these only if the inspection above shows old Creating/Error containers.
# destroy_named_containers("hi-edge-server", statuses={"Error", "Creating"})
# destroy_named_containers("hi-edge-device", statuses={"Error", "Creating"})

# Full reset if you intentionally want to recreate everything from scratch.
# This also releases the public IPs attached to these containers.
#destroy_named_containers("hi-edge-device")
#destroy_named_containers("hi-edge-server")


No containers named hi-edge-server.
No containers named hi-edge-device.


[]

### 3. Setup Edge Server

---

1. Reserve testbed equiment for the Edge Server
2. Load and run the container for Edge Server (amd64)
3. Print out used public IP for the container

In [4]:
# Worker reservation for Edge Server
worker_name = "worker-09"

found = False
leaseslist = list_reservations(brief=True)
for lease_dict in leaseslist:
  if lease_dict["name"] == worker_name+"-lease" and lease_dict.get("status") == "ACTIVE":
    worker_reservation_id = lease_dict["reservation_id"]
    found = True
    break

if not found:
  worker_lease = reserve(
    { "type":"device", "name":worker_name, "duration": { "days":7, "hours":0 } }
  )
  worker_reservation_id = worker_lease["reservations"][0]["id"]

print(json.dumps(leaseslist,indent=4))


2026-07-21 19:01:23.123 | INFO     | chi.expeca:reserve:243 - reserving worker-09
2026-07-21 19:01:25.644 | INFO     | chi.expeca:wait_until_lease_status:138 - waiting 120 seconds for worker-09-lease with id 17bfb7d4-676c-4254-9190-2d11aae25b5f to become "ACTIVE"
2026-07-21 19:01:30.821 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-09-lease with id 17bfb7d4-676c-4254-9190-2d11aae25b5f is PENDING.
2026-07-21 19:01:35.884 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-09-lease with id 17bfb7d4-676c-4254-9190-2d11aae25b5f is PENDING.
2026-07-21 19:01:40.950 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-09-lease with id 17bfb7d4-676c-4254-9190-2d11aae25b5f is PENDING.
2026-07-21 19:01:46.005 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-09-lease with id 17bfb7d4-676c-4254-9190-2d11aae25b5f is PENDING.
2026-07-21 19:01:51.064 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-09-lease with id 

[
    {
        "name": "worker-08-lease",
        "id": "3552b00d-7a32-4ed7-8de7-33708a5e02e3",
        "reservation_id": "5de3e32e-be11-4cd0-8506-3062eabe2b6b",
        "status": "TERMINATED",
        "end_date": "2026-07-16T07:19:00.000000"
    },
    {
        "name": "worker-09-lease",
        "id": "828a2064-9aad-444d-bf44-a469b545881e",
        "reservation_id": "863b5f4a-0bf2-4cd6-8772-4e6979c77481",
        "status": "TERMINATED",
        "end_date": "2026-07-20T15:45:00.000000"
    },
    {
        "name": "worker-09-lease",
        "id": "9fdcf126-cd12-4d1b-9a76-871fd77b2ecf",
        "reservation_id": "80887ec6-2bff-425e-9e5f-29c6c65295e8",
        "status": "TERMINATED",
        "end_date": "2026-07-13T14:41:00.000000"
    },
    {
        "name": "worker-05-lease",
        "id": "a5d0344c-3a7a-4722-b0ff-44d9b0cc9c29",
        "reservation_id": "667676f0-a112-4742-afe4-110d14ffb3dd",
        "status": "ACTIVE",
        "end_date": "2026-07-22T22:08:00.000000"
    },
    {


In [5]:
# Edge Server public IP reservation, loading and running container

# 1. Check public IPs and select one
available_pub_ips = get_available_publicips()
pub_ip = available_pub_ips[1]
logger.info(f"Available public ips: {available_pub_ips}.")
logger.info(f"We choose {pub_ip} for this container.")

# 2. Check available interfaces of the worker
interfaces = list(get_worker_interfaces(worker_name).values())[0]
available_ifs = []
for interface in interfaces.keys():
  if len(interfaces[interface]['connections']) == 0:
    available_ifs.append(interface)
logger.info(f"Available interfaces on {worker_name}: {available_ifs}")

# 3. Run the container
publicnet = chi.network.get_network("serverpublic")
container_name = "hi-edge-server"
create_container_with_progress(
    container_name,
    {
        "image": EDGE_SERVER_IMAGE,
        "reservation_id": worker_reservation_id,
        "environment": {
            "DNS_IP":"8.8.8.8",
            "GATEWAY_IP":"130.237.11.97",
            "PASS":"expeca",
            "DEVICE": EDGE_SERVER_DEVICE
        },
        "mounts": [],
        "nets": [
            { "network" : publicnet['id'] },
        ],
        "labels": {
            "networks.1.interface":available_ifs[0],
            "networks.1.ip":pub_ip+"/27",
            "networks.1.gateway":"130.237.11.97",
            "capabilities.privileged":"true",
        },
    },
)
logger.success(f"created {container_name} container, reachable at {pub_ip}.")

edge_server_pub_ip = pub_ip
logger.success(f"Edge Server public IP: {edge_server_pub_ip}")


2026-07-21 19:02:20.837 | INFO     | __main__:<module>:6 - Available public ips: ['130.237.11.120', '130.237.11.121', '130.237.11.122', '130.237.11.126'].
2026-07-21 19:02:20.839 | INFO     | __main__:<module>:7 - We choose 130.237.11.121 for this container.
2026-07-21 19:02:31.775 | INFO     | __main__:<module>:15 - Available interfaces on worker-09: ['eno12409', 'eno12419', 'ens1']
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
2026-07-21 19:02:55.519 | SUCCESS  | __main__:create_container_with_progress:174 - hi-edge-server is running.
2026-07-21 19:02:55.521 | SUCCESS  | __main__:<module>:43 - created hi-edge-server container, reachable at 130.237.11.121.
2026-07-21 19:02:55.524 | SUCCESS  | __main__:<module>:46 - Edge Server public IP: 130.237.11.12

19:02:55 status: Running reason: None addresses: {}


In [6]:
# Inspect the edge-server container without relying on name lookup when duplicates exist.
inspect_named_container("hi-edge-server")

print("\nAll containers:")
for c in chi.container.list_containers():
    print_container_summary(c)


Containers named hi-edge-server:
name= hi-edge-server uuid= 73c35c68-de6e-4237-8da8-cff39c04d105 status= Running reason= None image= justin157/hi-framework-edge-server:cpu-amd64-001 addresses= {}
logs:
[2026-07-21 17:02:53] Appending custom DNS: 8.8.8.8
[2026-07-21 17:02:53] Setting default gateway to 130.237.11.97
[2026-07-21 17:02:53] Configuring SSH access
Starting OpenBSD Secure Shell server: sshd.
[2026-07-21 17:02:53] Starting edge_server.py
2026-07-21 17:02:55,615 - INFO - Deleted file: results/EdgeServer.log
2026-07-21 17:02:55,615 - INFO - Edge Server: Starting... (Device: cpu, Port: 8001)
INFO:     Started server process [1]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


All containers:
name= hi-edge-server-002 uuid= 1d5d8f37-02e2-41e9-af60-9eb954d91568 status= Deleted reason= Container failed to create correctly image= justin157/hi-framework-edge-server:cpu-amd64-002

In [ ]:
# WARNING: Destroy the edge-server container only when you intentionally want to restart it.
# Prefer the cleanup cell above for stale duplicates.
destroy_named_containers("hi-edge-server")

In [ ]:
# Only use this when reusing an already-running edge-server container from a previous cell/run.
# Keep this value aligned with the public IP printed by the edge-server creation cell.
edge_server_pub_ip = "EDGE_SERVER_PUBLIC_IP_FROM_RUNNING_CONTAINER"
print(edge_server_pub_ip)

### 4. Setup Edge Device

---

*Note: Edge servers public IP is passed as "edge_server_pub_ip" to Edge Device*

**Steps:**

1. Reserve testbed equiment for the Edge Device:
  - Option 1 = Raspberry Pi (amd64)
  - Option 2 = Worker node (arm64)
2. Load and run the container for Edge Device
  - Use same option as previously
3. Print out used public IP for the container

#### Option 1 = Raspberry Pi (arm64)

You can edit the following parameters
- Used worker = "worker_name"
- Used image = "image"

In [7]:
# Option 1 = Raspberry Pi (arm64)
# Edge Device worker reservation

worker_name = "worker-21"

found = False
leaseslist = list_reservations(brief=True)
for lease_dict in leaseslist:
  if lease_dict["name"] == worker_name+"-lease" and lease_dict.get("status") == "ACTIVE":
    worker_reservation_id = lease_dict["reservation_id"]
    found = True
    break

if not found:
  worker_lease = reserve(
    { "type":"device", "name":worker_name, "duration": { "days":7, "hours":0 } }
  )
  worker_reservation_id = worker_lease["reservations"][0]["id"]

print(json.dumps(leaseslist,indent=4))


[
    {
        "name": "worker-09-lease",
        "id": "17bfb7d4-676c-4254-9190-2d11aae25b5f",
        "reservation_id": "5f659f9e-048e-4e58-a69b-0201ebf53538",
        "status": "ACTIVE",
        "end_date": "2026-07-28T17:01:00.000000"
    },
    {
        "name": "worker-08-lease",
        "id": "3552b00d-7a32-4ed7-8de7-33708a5e02e3",
        "reservation_id": "5de3e32e-be11-4cd0-8506-3062eabe2b6b",
        "status": "TERMINATED",
        "end_date": "2026-07-16T07:19:00.000000"
    },
    {
        "name": "worker-09-lease",
        "id": "828a2064-9aad-444d-bf44-a469b545881e",
        "reservation_id": "863b5f4a-0bf2-4cd6-8772-4e6979c77481",
        "status": "TERMINATED",
        "end_date": "2026-07-20T15:45:00.000000"
    },
    {
        "name": "worker-09-lease",
        "id": "9fdcf126-cd12-4d1b-9a76-871fd77b2ecf",
        "reservation_id": "80887ec6-2bff-425e-9e5f-29c6c65295e8",
        "status": "TERMINATED",
        "end_date": "2026-07-13T14:41:00.000000"
    },
    {


In [8]:
# Option 1 = Raspberry Pi (arm64)
# Edge Device public IP reservation, loading and running container

# 1. Check public IPs and select one
available_pub_ips = get_available_publicips()
pub_ip = available_pub_ips[0]
logger.info(f"Available public ips: {available_pub_ips}.")
logger.info(f"We choose {pub_ip} for this container.")

# 2. Check available interfaces of the worker
interfaces = list(get_worker_interfaces(worker_name).values())[0]
available_ifs = []
for interface in interfaces.keys():
  if len(interfaces[interface]['connections']) == 0:
    available_ifs.append(interface)
logger.info(f"Available interfaces on {worker_name}: {available_ifs}")

# 3. Run the container
publicnet = chi.network.get_network("serverpublic")
container_name = "hi-edge-device"
create_container_with_progress(
    container_name,
    {
        "image": EDGE_DEVICE_ARM64_IMAGE,
        "reservation_id": worker_reservation_id,
        "environment": {
            "DNS_IP":"8.8.8.8",
            "GATEWAY_IP":"130.237.11.97",
            "PASS":"expeca",
            "EDGE_SERVER_IP": edge_server_pub_ip
        },
        "mounts": [],
        "nets": [
            { "network" : publicnet['id'] },
        ],
        "labels": {
            "networks.1.interface":available_ifs[0],
            "networks.1.ip":pub_ip+"/27",
            "networks.1.gateway":"130.237.11.97",
            "capabilities.privileged":"true",
        },
    },
)
edge_device_pub_ip = pub_ip
logger.success(f"created {container_name} container, reachable at {edge_device_pub_ip}.")
logger.success(f"Edge Device public IP: {edge_device_pub_ip}")


2026-07-21 19:03:17.435 | INFO     | __main__:<module>:7 - Available public ips: ['130.237.11.120', '130.237.11.122', '130.237.11.126'].
2026-07-21 19:03:17.436 | INFO     | __main__:<module>:8 - We choose 130.237.11.120 for this container.
2026-07-21 19:03:29.182 | INFO     | __main__:<module>:16 - Available interfaces on worker-21: ['enx000acd47c9fc', 'enx000acd47cdaf', 'enx000acd47cdb0', 'eth0']
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
2026-07-21 19:03:47.709 | SUCCESS  | __main__:create_container_with_progress:174 - hi-edge-device is running.
2026-07-21 19:03:47.711 | SUCCESS  | __main__:<module>:45 - created hi-edge-device container, reachable at 130.237.11.120.
2026-07-21 19:03:47.713 | SUCCESS  | __main__:<module>:46 - Edge Device public IP

19:03:47 status: Running reason: None addresses: {'e9666f9c-1ef6-4be7-81cd-32373cfacc28': [{'addr': '192.168.105.226', 'version': 4, 'port': '417e9e3b-0bd7-4707-80c0-3926bbf2985d'}]}


#### Option 2 = Worker node (amd64)

You can edit the following parameters
- Used worker = "worker_name"
- Used image = "image"

In [9]:
# Option 2 = Worker node (amd64)
# Edge Device worker reservation

worker_name = "worker-09"

found = False
leaseslist = list_reservations(brief=True)
for lease_dict in leaseslist:
  if lease_dict["name"] == worker_name+"-lease" and lease_dict.get("status") == "ACTIVE":
    worker_reservation_id = lease_dict["reservation_id"]
    found = True
    break

if not found:
  worker_lease = reserve(
    { "type":"device", "name":worker_name, "duration": { "days":7, "hours":0 } }
  )
  worker_reservation_id = worker_lease["reservations"][0]["id"]

print(json.dumps(leaseslist,indent=4))


[
    {
        "name": "worker-08-lease",
        "id": "3552b00d-7a32-4ed7-8de7-33708a5e02e3",
        "reservation_id": "5de3e32e-be11-4cd0-8506-3062eabe2b6b",
        "status": "ACTIVE",
        "end_date": "2026-07-16T07:19:00.000000"
    },
    {
        "name": "worker-09-lease",
        "id": "828a2064-9aad-444d-bf44-a469b545881e",
        "reservation_id": "863b5f4a-0bf2-4cd6-8772-4e6979c77481",
        "status": "ACTIVE",
        "end_date": "2026-07-20T15:45:00.000000"
    },
    {
        "name": "worker-09-lease",
        "id": "9fdcf126-cd12-4d1b-9a76-871fd77b2ecf",
        "reservation_id": "80887ec6-2bff-425e-9e5f-29c6c65295e8",
        "status": "TERMINATED",
        "end_date": "2026-07-13T14:41:00.000000"
    }
]


In [10]:
# Option 2 = Worker node (amd64)
# Edge Device public IP reservation, loading and running container

# 1. Check public IPs and select one
available_pub_ips = get_available_publicips()
pub_ip = available_pub_ips[0]
logger.info(f"Available public ips: {available_pub_ips}.")
logger.info(f"We choose {pub_ip} for this container.")

# 2. Check available interfaces of the worker
interfaces = list(get_worker_interfaces(worker_name).values())[0]
available_ifs = []
for interface in interfaces.keys():
  if len(interfaces[interface]['connections']) == 0:
    available_ifs.append(interface)
logger.info(f"Available interfaces on {worker_name}: {available_ifs}")

# 3. Run the container
publicnet = chi.network.get_network("serverpublic")
container_name = "hi-edge-device"
create_container_with_progress(
    container_name,
    {
        "image": EDGE_DEVICE_AMD64_IMAGE,
        "reservation_id": worker_reservation_id,
        "environment": {
            "DNS_IP":"8.8.8.8",
            "GATEWAY_IP":"130.237.11.97",
            "PASS":"expeca",
            "DEVICE": EDGE_DEVICE_DEVICE,
            "EDGE_SERVER_IP": edge_server_pub_ip
        },
        "mounts": [],
        "nets": [
            { "network" : publicnet['id'] },
        ],
        "labels": {
            "networks.1.interface":available_ifs[0],
            "networks.1.ip":pub_ip+"/27",
            "networks.1.gateway":"130.237.11.97",
            "capabilities.privileged":"true",
        },
    },
)
logger.success(f"created {container_name} container, reachable at {pub_ip}.")
edge_device_pub_ip = pub_ip
logger.success(f"Edge Device public IP: {edge_device_pub_ip}")


2026-07-13 18:11:51.198 | INFO     | __main__:<module>:7 - Available public ips: ['130.237.11.117', '130.237.11.121', '130.237.11.122', '130.237.11.123', '130.237.11.125', '130.237.11.126'].
2026-07-13 18:11:51.199 | INFO     | __main__:<module>:8 - We choose 130.237.11.117 for this container.
2026-07-13 18:12:02.349 | INFO     | __main__:<module>:16 - Available interfaces on worker-09: ['eno12409', 'eno12419', 'ens1']
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
2026-07-13 18:12:16.033 | SUCCESS  | __main__:create_container_with_progress:173 - hi-edge-device is running.
2026-07-13 18:12:16.034 | SUCCESS  | __main__:<module>:45 - created hi-edge-device container, reachable at 130.237.11.117.
2026-07-13 18:12:16.034 | SUCCESS  | __main__:<module>:47 - 

18:12:16 status: Running reason: None addresses: {}


In [9]:
# Inspect the edge-device container without relying on name lookup when duplicates exist.
inspect_named_container("hi-edge-device")

print("\nAll containers:")
for c in chi.container.list_containers():
    print_container_summary(c)


Containers named hi-edge-device:
name= hi-edge-device uuid= b718da48-495d-4850-8cc7-a05dffd3d236 status= Running reason= None image= justin157/hi-framework-edge-device:cpu-arm64-001 addresses= {'e9666f9c-1ef6-4be7-81cd-32373cfacc28': [{'addr': '192.168.105.226', 'version': 4, 'port': '417e9e3b-0bd7-4707-80c0-3926bbf2985d'}]}
logs:
[2026-07-21 17:03:44] Appending custom DNS: 8.8.8.8
[2026-07-21 17:03:44] Setting default gateway to 130.237.11.97
[2026-07-21 17:03:44] Configuring SSH access
Starting OpenBSD Secure Shell server: sshd.
[2026-07-21 17:03:44] Starting edge_device.py


All containers:
name= hi-edge-server-002 uuid= 1d5d8f37-02e2-41e9-af60-9eb954d91568 status= Deleted reason= Container failed to create correctly image= justin157/hi-framework-edge-server:cpu-amd64-002 addresses= {}
name= hi-edge-server8 uuid= 2ee44f7f-f6cd-43a7-9315-96fddca4c2da status= Deleted reason= Container failed to create correctly image= justin157/hi-framework-edge-server:cpu-amd64-001 addresses= {}
name

### Connecting to the Edge Device and Edge Server using SSH

---

Now the setup should be completed and you can use the external controller scripts to run the experiments.
- External controller scripts are found in `app/external_controllers/`
- SSH is not required if using public IP's.
- Chameleon web interface allows easy monitoring and control of the containers.

You can ssh into the containers with the following command:
```
ssh root@[public_ip_address]
```

### Running the Benchmark

This notebook only creates and checks the ExPECA containers. The experiment is driven from the local benchmark runner, not from `/app/start.sh` inside the edge-device container.

After both containers are running, copy the printed public IPs into `config/experiment.env`:

```env
EDGE_SERVER_IP=...
EDGE_DEVICE_IP=...
```

Then run the thesis reproduction from the repository root:

```bash
.venv/bin/python src/run_thesis_reproduction.py --dry-run
.venv/bin/python src/run_thesis_reproduction.py
```

Results and plots are written under the configured `results/` output folder.
